# 03 - Generators and Provider System

> **When to use**: When you need to choose a data generation engine (base/faker/mimesis), or understand the capabilities of 31 generators.
>
> **Core concept**: sqlseed supports 3 Providers, descending by richness: mimesis (default recommended) → faker → base (zero dependency).

## Applicable Scenarios

- CI/CD environment without extra dependencies → use `base`
- Need rich localized data (Chinese names, addresses, etc.) → use `mimesis`
- Need specific format data (e.g., SSN, license plate) → check 31 generators
- Want to develop custom generators → implement `DataProvider` Protocol

## What You Will Learn

- 31 built-in generator types
- Capability differences and fallback strategy of 3 Providers
- locale localization support
- Custom Provider development

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| **→ 03** | **Generators and Provider System** | **Generators** | **01** |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Generator Hub | `src/sqlseed/generators/registry.py` | `GeneratorRegistry` |

> Corresponding architecture diagram: [§4 Data Generation Layer Architecture](../docs/architecture.zh-CN.md#4-数据生成层架构)

## 1. See the Effect First — Three Providers Comparison

Same table, same column, the data quality difference between Providers is clear at a glance:

In [2]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    sep = "=" * 70
    print()
    print(sep)
    print(f"  Provider: {provider_name}")
    print(sep)
    for row in rows:
        addr = str(row.get("address", "N/A"))[:40]
        print(f"  name={row.get('name', 'N/A'):<20s} email={row.get('email', 'N/A'):<30s}")
        print(f"  phone={row.get('phone', 'N/A'):<20s} address={addr}")


  Provider: base
  name=Sharon Martin        email=cynthia.white364@sample.dev   
  phone=843-338-9697         address=8483 Maple Dr, Springfield, PA
  name=Ryan Adams           email=margaret.baker355@example.com 
  phone=308-867-1045         address=3719 Maple Dr, Portland, PA

  Provider: faker
  name=Micheal Stewart      email=carrieramsey@example.org      
  phone=367.580.7384x0668    address=27294 Ronald Cape, Reeseland, CA 15745
  name=Misty Stanton        email=heather76@example.com         
  phone=443.851.4460         address=59695 White Courts Suite 343, Christinev

  Provider: mimesis
  name=Fidel Camacho        email=dig1888@example.org           
  phone=+17698016227         address=504 Coldspring Walk
  name=Josef Estrada        email=clusters1817@yahoo.com        
  phone=+13649258871         address=561 Soule Extension


- **base**: no external dependencies, generates random strings — suitable for CI
- **faker**: rich community ecosystem, many methods — suitable for dev debugging
- **mimesis**: best localization, highest performance — **default recommendation**

Let's break down each generator in detail.

## 2. 31 Generators Overview

sqlseed has 31 built-in generators, divided into three categories:

### Basic Types (7)

| Generator | Description | Key Params |
|--------|------|----------|
| `string` | random string | min_length, max_length, charset |
| `integer` | integer | min_value, max_value |
| `float` | float | min_value, max_value, precision |
| `boolean` | boolean | - |
| `bytes` | binary data | length |
| `json` | JSON data | - |
| `choice` | enum choice | choices |

### Semantic Types (17)

| Generator | Description | Example |
|--------|------|----------|
| `name` | name | 张三 / John Smith |
| `first_name` | first name | 伟 / John |
| `last_name` | last name | 王 / Smith |
| `username` | username | user_3847 |
| `email` | email | test@example.com |
| `phone` | phone | +1-555-0123 |
| `address` | address | 123 Main St |
| `city` | city | Beijing / New York |
| `country` | country | China / United States |
| `state` | state/province | California |
| `zip_code` | zip code | 10001 |
| `company` | company | Acme Inc |
| `job_title` | job title | Software Engineer |
| `url` | URL | https://example.com |
| `ipv4` | IP address | 192.168.1.1 |
| `uuid` | UUID | 550e8400-e29b-41d4... |
| `country_code` | country code | CN / US |

### Time/Text Types (7)

| Generator | Description | Example |
|--------|------|----------|
| `date` | date | 2024-01-15 |
| `datetime` | datetime | 2024-01-15 10:30:00 |
| `timestamp` | timestamp | 1705312200 |
| `text` | long text | Lorem ipsum... |
| `sentence` | sentence | The quick brown fox... |
| `password` | password | k8Xf2mPq |
| `pattern` | regex generation | PRJ-000123 |

| Feature | BaseProvider | FakerProvider | MimesisProvider |
|------|:-----------:|:------------:|:--------------:|
| Dependency | None | faker | mimesis |
| Localization | ❌ | ✅ (en_US, zh_CN...) | ✅ (en, zh...) |
| Semantic Quality | Basic random | High | Highest |
| Speed | Fastest | Medium | Fast |
| Install | Default | `pip install sqlseed[faker]` | `pip install sqlseed[mimesis]` |

## 3. 6 Representative Generators Deep Demo

### 3.1 email — Semantic Inference

The `email` generator generates different styles of email addresses based on locale.

In [3]:
from sqlseed import preview

rows = preview(str(db_path), table="members", count=3, provider="mimesis")
for row in rows:
    print(f"email: {row['email']}")

email: hepatitis1900@duck.com
email: ppc1955@duck.com
email: switches2071@example.com


### 3.2 pattern — Regex Generation

The `pattern` generator uses the `rstr` library to generate data from regex, suitable for fixed-format IDs.

In [4]:
rows = preview(
    str(db_path),
    table="projects",
    count=5,
    columns={
        "project_no": {"type": "pattern", "regex": "PRJ-\\d{6}"},
    },
)
for row in rows:
    print(f"project_no: {row['project_no']}")

project_no: PRJ-703549
project_no: PRJ-948784
project_no: PRJ-852110
project_no: PRJ-090002
project_no: PRJ-385638


### 3.3 choice — Enum Selection

The `choice` generator randomly selects from given options, suitable for finite sets like status, type, etc.

In [5]:
rows = preview(
    str(db_path),
    table="tasks",
    count=5,
    columns={
        "priority": {"type": "choice", "choices": [1, 2, 3, 4]},
        "status": {"type": "choice", "choices": [0, 1, 2, 3]},
    },
)
for row in rows:
    print(f"priority: {row['priority']}, status: {row['status']}")

priority: 2, status: 3
priority: 1, status: 3
priority: 3, status: 2
priority: 3, status: 3
priority: 2, status: 2


### 3.4 null_ratio — Null Control

The `null_ratio` parameter controls the proportion of nulls generated (0.0-1.0).

In [6]:
import sqlite3

from sqlseed import ColumnConfig

# null_ratio needs to be passed via ColumnConfig object, using connect() API
with connect(str(db_path)) as orch:
    result = orch.fill_table("members", count=20, clear_before=True,
        column_configs=[
            ColumnConfig(name="phone", generator="phone", null_ratio=0.3),
            ColumnConfig(name="address", generator="address", null_ratio=0.5),
        ])


conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT phone, address FROM members").fetchall()
phone_null = sum(1 for r in rows if r[0] is None)
addr_null = sum(1 for r in rows if r[1] is None)
print(f"phone null: {phone_null}/20 ({phone_null/20*100:.0f}%)")
print(f"address null: {addr_null}/20 ({addr_null/20*100:.0f}%)")
conn.close()

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

phone null: 5/20 (25%)
address null: 15/20 (75%)


### 3.5 faker.name — Provider Switching

Switch Provider at column level via the `provider` parameter. `native_faker_method` can call Faker's native methods.

In [7]:
rows = preview(
    str(db_path),
    table="members",
    count=3,
    provider="faker",
    locale="zh_CN",
)
for row in rows:
    print(f"name: {row['name']}, email: {row['email']}")
print("\nFaker + zh_CN locale generates Chinese names and emails")

name: 陈淑兰, email: wei22@example.net
name: 潘斌, email: pingmo@example.org
name: 谭秀云, email: rsong@example.net

Faker + zh_CN locale generates Chinese names and emails


### 3.6 mimesis.address — Mimesis Comparison

Mimesis is the default Provider with the best localization support. Note the locale format differences:
- Faker: `en_US`, `zh_CN` (with underscore)
- Mimesis: `en`, `zh` (short code)

In [8]:
rows = preview(
    str(db_path),
    table="members",
    count=3,
    provider="mimesis",
    locale="zh",
)
for row in rows:
    print(f"name: {row['name']}, address: {row.get('address', 'N/A')}")
print("\nMimesis + zh locale generates Chinese names and addresses")

name: 恒世 上官, address: 北湖三条1302号
name: 晨濡 宗政, address: 柿铺六条325号
name: 虹 汲, address: 七里河侧路393号

Mimesis + zh locale generates Chinese names and addresses


## 4. Provider Fallback Strategy

When the preferred Provider is unavailable, sqlseed auto-falls back:

```
mimesis → faker → base
```

- If `mimesis` is not installed, auto-fallback to `faker`
- If `faker` is also not installed, fallback to `base`
- `base` is always available, zero dependency

Fallback is silent, no errors, but generation quality decreases.

In [9]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    print(f"\n--- {provider_name} ---")
    for row in rows:
        print(f"  name={row['name']}, email={row['email']}")


--- base ---
  name=Laura Adams, email=linda.gomez552@mail.net
  name=Joseph Carter, email=george.robinson356@mail.net

--- faker ---
  name=Joshua Zimmerman, email=allen08@example.com
  name=Scott Austin, email=mguzman@example.com

--- mimesis ---
  name=Sylvie Dillon, email=slow1813@outlook.com
  name=Elayne Evans, email=committees2087@duck.com


## 5. locale and seed

### locale Settings

| Provider | Supported locale Format | Example |
|----------|-------------------|------|
| mimesis | short code | `en`, `zh`, `ja`, `de` |
| faker | underscore code | `en_US`, `zh_CN`, `ja_JP` |
| base | no localization | - |

### seed Reproducibility

In [10]:
rows1 = preview(str(db_path), table="members", count=3, seed=42)
rows2 = preview(str(db_path), table="members", count=3, seed=42)

names1 = [r['name'] for r in rows1]
names2 = [r['name'] for r in rows2]

print(f"Run 1: {names1}")
print(f"Run 2: {names2}")
print(f"Reproducible: {names1 == names2}")

Run 1: ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
Run 2: ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
Reproducible: True


## 🆕 bytes / json / timestamp Generators

These three generators are for special data types:

In [11]:
with sqlseed.connect(str(db_path)) as orch:
    preview_bytes = orch.preview_table("organizations", count=3, columns={
        "description": {"generator": "bytes"},
    })
    print("bytes generator (raw bytes):")
    for row in preview_bytes:
        val = row.get('description')
        print(f"  description type: {type(val).__name__}, len: {len(val) if val else 0}")

preview_json = sqlseed.preview(str(db_path), table="organizations", count=2, columns={
    "description": {"generator": "json"}
})
print("\njson generator:")
for row in preview_json:
    print(f"  {row.get('description', 'N/A')}")

preview_ts = sqlseed.preview(str(db_path), table="organizations", count=2, columns={
    "created_at": {"generator": "timestamp"}
})
print("\ntimestamp generator:")
for row in preview_ts:
    print(f"  created_at: {row.get('created_at')}")

bytes generator (raw bytes):
  description type: bytes, len: 16
  description type: bytes, len: 16
  description type: bytes, len: 16

json generator:
  {"id": 839617, "name": "Orval Hunter", "active": true}
  {"id": 181944, "name": "Maximo King", "active": false}

timestamp generator:
  created_at: 1786317696
  created_at: 1778608156


## 🔧 Underlying Provider Native Methods

sqlseed's Providers wrap the Faker and Mimesis libraries. Advanced users can access the underlying library's native methods via ProviderRegistry to get data types not covered by built-in generators (e.g., `license_plate`, `credit_card_number`, `food.fruit`, etc.).

In [12]:
from sqlseed.generators.registry import ProviderRegistry

# Underlying Provider can call native methods directly
# This is advanced usage, generally columns={} config is sufficient
registry = ProviderRegistry()

# Faker native methods
registry.ensure_provider("faker")
faker_provider = registry.get("faker")
faker_provider.set_locale("en_US")
faker_obj = getattr(faker_provider, "_faker", None)

print("Faker native method examples:")
print(f"  company_suffix: {faker_obj.company_suffix()}")
print(f"  catch_phrase:   {faker_obj.catch_phrase()}")
print(f"  bs:             {faker_obj.bs()}")
print(f"  license_plate:  {faker_obj.license_plate()}")

# Mimesis native methods
registry.ensure_provider("mimesis")
mimesis_provider = registry.get("mimesis")
generic_obj = getattr(mimesis_provider, "_generic", None)

print("\nMimesis native method examples:")
print(f"  text.word:      {generic_obj.text.word()}")
print(f"  person.title:   {generic_obj.person.title()}")
print(f"  food.fruit:     {generic_obj.food.fruit()}")
print(f"  science.metric: {generic_obj.science.metric_prefix()}")

Faker native method examples:
  company_suffix: and Sons
  catch_phrase:   Face-to-face national analyzer
  bs:             cultivate intuitive e-business
  license_plate:  934 CAV

Mimesis native method examples:
  text.word:      technique
  person.title:   LL.D
  food.fruit:     Honeydew
  science.metric: giga


## 📦 ProviderRegistry Complete API

ProviderRegistry manages all data generation Providers, supporting registration, query, setting defaults, etc.

In [13]:
from sqlseed.generators.registry import ProviderRegistry

registry = ProviderRegistry()
print(f"Default Provider: {registry.default_name}")
print(f"Available Providers: {registry.available_providers}")

base = registry.get("base")
print(f"\nBaseProvider: name={base.name}")

registry.ensure_provider("faker")
print(f"After loading FakerProvider: {registry.available_providers}")

registry.ensure_provider("mimesis")
print(f"After loading MimesisProvider: {registry.available_providers}")

registry.set_default("faker")
print(f"Switch default to faker: default_name={registry.default_name}")

Default Provider: base
Available Providers: ['base']

BaseProvider: name=base
After loading FakerProvider: ['base', 'faker']
After loading MimesisProvider: ['base', 'faker', 'mimesis']
Switch default to faker: default_name=faker


## 🔄 Provider Switching

Switch generation engine via the `provider` parameter of `preview()` / `fill()`. Different Providers have significant differences in data quality and localization support:

In [14]:
# Provider switching via preview/fill's provider parameter
# Quality differences for same data type across different Providers:

print("Provider comparison — same column, different engines:\n")
print(f"{'Provider':<10s}  {'name':<25s}  {'email':<30s}")
print("-" * 68)

for prov in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=prov)
    for row in rows:
        print(f"{prov:<10s}  {row['name']:<25s}  {row['email']:<30s}")
    print()

print("Recommendation: use faker for dev debugging, base for CI, mimesis for production data")

Provider comparison — same column, different engines:

Provider    name                       email                         
--------------------------------------------------------------------
base        Amanda Robinson            jeffrey.gomez439@demo.io      
base        Donna Williams             betty.nelson423@demo.io       

faker       Olivia Quinn               susan65@example.net           
faker       Patrick Tyler              nreeves@example.net           

mimesis     Wanetta Pena               duty1810@gmail.com            
mimesis     Weldon Harper              effectiveness1944@example.com 

Recommendation: use faker for dev debugging, base for CI, mimesis for production data


## 6. Summary

| Key Point | Description |
|------|------|
| 31 generators | Basic 7 + Semantic 17 + Time/Text 7 |
| 3 Providers | mimesis (default) > faker > base |
| Auto fallback | Silent fallback when deps missing, no error |
| locale | mimesis uses short codes, faker uses underscore codes |
| seed | Reproducible when set, suitable for testing |
| null_ratio | 0.0-1.0 controls null proportion |

**Next**: [04-database-advanced.ipynb](04-database-advanced.ipynb) — Database Layer and Multi-table

In [15]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
